# Step 1: Get Features From Multiple Datasets
- Using [pybiber](https://pypi.org/project/pybiber/)

In [2]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

from transformers import logging
logging.set_verbosity_error()

In [3]:
import pybiber as pb
import polars as pl
import os
import numpy as np
import random
import torch
from scipy.stats import zscore
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import gc
import importlib.resources as resources

from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, matthews_corrcoef
from scipy.stats import pearsonr, spearmanr

In [4]:
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR = 'biberOutputs'
SAMPLE_SIZE = 10
batch_idx = 0

In [5]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-deberta-v3-small", # low capacity
    # "cross-encoder/nli-MiniLM2-L6-H768",
    # "MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary",
    # "MoritzLaurer/roberta-base-zeroshot-v2.0-c", # not a large amount of improvement after adding these models so excluding them
    "typeform/distilbert-base-uncased-mnli", # medium capacity
    "valhalla/distilbart-mnli-12-3", # higher capacity
]

# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": {
        "informational, dense, precise": -1,
        "involved, interactive, affective": 1
    },
    "factor_2": {
        "non-narrative, expository, informational": -1,
        "narrative, event-focused, storytelling": 1
    },
    "factor_3": {
        "situation-dependent, context-bound, implicit": -1,
        "explicit, context-independent, elaborated": 1
    },
    "factor_4": {
        "non-persuasive, non-argumentative, neutral": -1,
        "persuasive, argumentative, modalized": 1
    },
    "factor_5": {
        "non-abstract, concrete, human-centered": -1,
        "abstract, impersonal, technical": 1
    },
    "factor_6": {
        "compressed, dense, clause-poor": -1,
        "elaborated, expanded, clause-rich": 1
    }
}

# Using different prompt templates increases robustness. Ability to modify for different factors. 
TEMPLATES = { # [default, default, more specialized, even more specialized, most specialised], moving from neutral to theory-aware phrasing
    "factor_1": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_2": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_3": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_4": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_5": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_6": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."]
}


In [6]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [7]:
# Load all data.
list_of_dfs = []
for folder in os.listdir(f'./{FILE_PATH}'):
    if os.path.isdir(f'./{FILE_PATH}/{folder}'):
        for file in os.listdir(f'./{FILE_PATH}/{folder}'):
            if file.endswith(".csv") and 'train' in file:
                temp_file_path = f'./{FILE_PATH}/{folder}/{file}'
                temp_tag = file.replace('_train.csv', '')
                temp_df = pl.read_csv(temp_file_path)
                temp_df = (
                    temp_df
                    .with_row_index("index_num") # , offset=1) if you want to start index from 1
                    .with_columns(
                        (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
                    )).select(['text', 'doc_id'])
                list_of_dfs.append(temp_df)

combined = pl.concat(list_of_dfs, how="vertical")
assert combined.select(pl.col("doc_id").n_unique()).item() == combined.height, "There should be no duplicates in the dataset."

In [8]:
combined

text,doc_id
str,str
"""After taking written informed …","""augmentedClinicalNotes_0"""
"""A 59-year-old man presented to…","""augmentedClinicalNotes_1"""
"""A 36 year-old gentleman presen…","""augmentedClinicalNotes_2"""
"""1: A-39-years-old male referre…","""augmentedClinicalNotes_3"""
"""The patient was a 7-year-old g…","""augmentedClinicalNotes_4"""
"""A previously healthy 6-year-ol…","""augmentedClinicalNotes_5"""
"""An 8-year-old Asian female chi…","""augmentedClinicalNotes_6"""
"""Patient IM, a 43-year-old Fren…","""augmentedClinicalNotes_7"""
"""A 40-year-old man with no sign…","""augmentedClinicalNotes_8"""


In [ ]:
# Remove invalid data.
temp_df = combined.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts Before Empty String Removal: {len(temp_df)}")

temp_df = combined.with_columns(pl.col("text").str.strip_chars().alias("text")).filter(pl.col("text").is_not_null() & (pl.col("text") != ""))

temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts After Empty String Removal: {len(temp_df)}")

In [ ]:
# Randomly sample from dataframe.
temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
temp_df = temp_df.to_pandas()
temp_df = (temp_df.groupby("tag")).apply(lambda x: x.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE + batch_idx)) ; batch_idx += 1 # Increase batch_idx every time to ensure new random samples.
temp_df = pl.from_pandas(temp_df)


In [ ]:
# Set up df for use.
df = pl.DataFrame({
    "doc_id": temp_df['doc_id'].to_list(),
    "text": temp_df['text'].to_list()
})

# Light preprocessing to strip extra whitespace.
df = df.with_columns(
    pl.col("text")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
    .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
)

pybiber_pipeline = pb.PybiberPipeline(model="en_core_web_sm")
features, tokens = pybiber_pipeline.run(df, return_tokens=True)
features = features.with_columns(pl.col("doc_id").str.split("_").list.get(0).alias("category"))
# Full feature list can be found here: https://browndw.github.io/pybiber/feature-categories.html
print(f" -------- Features-------- ")
print(features)

# Statistical analysis and visualization
analyzer = pb.BiberAnalyzer(features, id_column='category')

# Multi-Dimensional Analysis - see https://browndw.github.io/pybiber/biber-analyzer.html#comparison-with-bibers-original-dimensions for factor mapping
# Explanation of the factor mapping to dimensions can be found here: https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html
'''
Factor 1: Involved vs. Informational Production (negative to positive)
Factor 2: Narrative vs. Non-narrative Concerns (negative to positive)
Factor 3: Explicit vs. Situation-dependent Reference (negative to positive)
Factor 4: Overt Expression of Persuasion (negative to positive)
Factor 5: Abstract vs. Non-abstract Information (negative to positive)
Factor 6: On-line Informational Elaboration (negative to positive)
'''

analyzer.mda_biber()
print(f" -------- MDA Loadings -------- ")
print(analyzer.mda_loadings)
print(f" -------- MDA Dimension Scores -------- ")
print(analyzer.mda_dim_scores)

In [ ]:
os.makedirs(f"./{OUTPUT_DIR}/", exist_ok=True)

def flatten_for_csv(df):
    # Work on a copy to avoid modifying original.
    df_flat = df.clone()
    for c, dtype in zip(df_flat.columns, df_flat.dtypes):
        if dtype == pl.List:
            # Join list elements into string with commas.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, [",".join(map(str, x)) if x is not None else "" for x in df_flat[c]])
            )
        elif dtype == pl.Struct:
            # Convert struct to string representation.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, df_flat[c].cast(pl.Utf8))
            )
    return df_flat

# Get Z-Scores from Biber analysis.
biber_dimensions = (analyzer.mda_dim_scores).to_pandas()
# Get factor columns.
factor_cols = [c for c in biber_dimensions.columns if c.startswith("factor")]
biber_dimensions[factor_cols] = biber_dimensions[factor_cols].apply(zscore)
for c in factor_cols:
    biber_dimensions[f"{c}_label"] = biber_dimensions[c] > 0
biber_dimensions = pl.from_pandas(biber_dimensions)
print(biber_dimensions)

In [ ]:
# Write CSV files.
flatten_for_csv(biber_dimensions).write_csv(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(analyzer.mda_loadings).write_csv(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)

# Write JSON files.
analyzer.mda_loadings.write_json(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.json")
biber_dimensions.write_json(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.json")

In [ ]:
temp_df = temp_df.to_pandas()
texts = temp_df['text'].values.tolist()
doc_ids = temp_df['doc_id'].values.tolist()

In [ ]:
assert len(texts) == len(doc_ids), "texts and doc_ids are not of the same length."

In [ ]:
rows = []
for model_name in ZERO_SHOT_MODELS:
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=DEVICE
    )
    for i in range(len(texts)):
        temp_factors = {'model_name': model_name, 'doc_id': doc_ids[i]}
        for factor, description in BIBER_LABEL_MAP.items():
            final_dimension_score = 0
            for template in TEMPLATES[factor]:
                outputs = classifier(
                    texts[i],
                    candidate_labels=list(description.keys()),
                    hypothesis_template=template,
                    batch_size=8,
                    multi_label = True
                )
                for label, score in zip(outputs['labels'], outputs['scores']):
                    final_dimension_score += description[label] * score
            final_dimension_score = final_dimension_score / len(TEMPLATES)
            temp_factors[factor] = final_dimension_score
        rows.append(temp_factors)
    # Free memory.
    del classifier
    torch.cuda.empty_cache()
    gc.collect()

    # break

df = pd.DataFrame(rows)


In [ ]:
dimension_labels_verbose = {
    "factor_1_label": "informational_vs_involved",
    "factor_2_label": "non-narrative_vs_narrative",
    "factor_3_label": "situation-dependent_vs_explicit",
    "factor_4_label": "non-persuasive_vs_persuasive",
    "factor_5_label": "non-abstract_vs_abstract",
    "factor_6_label": "compressed_vs_elaborated"
}

# Weighted Averages
# model_weights = {
#     "cross-encoder/nli-deberta-v3-small": 0.25,
#     "typeform/distilbert-base-uncased-mnli": 0.25,
#     "valhalla/distilbart-mnli-12-3": 0.50
# }

# weighted_rows = []
# for model_name, group in df.groupby("model_name"):
#     weight = model_weights.get(model_name, 1.0)  # default 1 if not in dict
#     weighted_group = group.copy()
#     for factor in ["factor_1", "factor_2", "factor_3", "factor_4", "factor_5", "factor_6"]:
#         weighted_group[factor] = weighted_group[factor] * weight
#     weighted_rows.append(weighted_group)

# weighted_df = pd.concat(weighted_rows)
# mean_scores = weighted_df.drop(columns='model_name').groupby("doc_id").sum().reset_index()

# Simple averaging will prevent over-confidence. 
mean_scores = df.drop(columns='model_name').groupby("doc_id").mean().reset_index()


# # Get factor columns.
factor_cols = [c for c in mean_scores.columns if c.startswith("factor")]
# # Get Z-Scores from zero-shot analysis.
mean_scores[factor_cols] = mean_scores[factor_cols].apply(zscore)
for c in factor_cols:
    mean_scores[f"{c}_label"] = mean_scores[c] > 0
mean_scores = pl.from_pandas(mean_scores)

results_cont = {}
for f in ["factor_1", "factor_2", "factor_3", "factor_4", "factor_5", "factor_6"]:
    pearson = pearsonr(biber_dimensions[f], mean_scores[f])[0]
    spearman = spearmanr(biber_dimensions[f], mean_scores[f])[0]
    mse = mean_squared_error(biber_dimensions[f], mean_scores[f])
    rmse = root_mean_squared_error(biber_dimensions[f], mean_scores[f])
    mae = mean_absolute_error(biber_dimensions[f], mean_scores[f])
    results_cont[f] = {"pearson": pearson, "spearman": spearman, "MSE": mse, "RMSE": rmse, "MAE": mae}
continuous_df = pd.DataFrame(results_cont).T
continuous_df = continuous_df.rename(index={
    "factor_1": "informational_vs_involved",
    "factor_2": "non-narrative_vs_narrative",
    "factor_3": "situation-dependent_vs_explicit",
    "factor_4": "non-persuasive_vs_persuasive",
    "factor_5": "non-abstract_vs_abstract",
    "factor_6": "compressed_vs_elaborated"
})
continuous_df['dimension'] = continuous_df.index
print(pl.from_pandas((continuous_df)))

results_bin = {}
for f in ["factor_1_label", "factor_2_label", "factor_3_label", "factor_4_label", "factor_5_label", "factor_6_label"]:
    acc = accuracy_score(biber_dimensions[f], mean_scores[f])
    prec = precision_score(biber_dimensions[f], mean_scores[f])
    rec = recall_score(biber_dimensions[f], mean_scores[f])
    f1 = f1_score(biber_dimensions[f], mean_scores[f])
    kappa = cohen_kappa_score(biber_dimensions[f], mean_scores[f])
    mcc = matthews_corrcoef(biber_dimensions[f], mean_scores[f])
    results_bin[f] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "kappa": kappa, "MCC": mcc}

classification_df = pd.DataFrame(results_bin).T
classification_df = classification_df.rename(index={
    "factor_1_label": "informational_vs_involved",
    "factor_2_label": "non-narrative_vs_narrative",
    "factor_3_label": "situation-dependent_vs_explicit",
    "factor_4_label": "non-persuasive_vs_persuasive",
    "factor_5_label": "non-abstract_vs_abstract",
    "factor_6_label": "compressed_vs_elaborated"
})
classification_df['dimension'] = classification_df.index
print(pl.from_pandas((classification_df)))